# Sampling Design States: Complete Validation

This notebook demonstrates ProcessBehavior's handling of all six Sampling Design States (SDS 1-6), following Wheeler's Variance Analysis System (VAS) methodology.

## The VAS Problem Formulation

A rigorous VAS formulation includes:
- **Unit of Analysis**: The fundamental entity being measured
- **Response**: The measurement variable
- **Factors**: Grouping variables defining rational subgroups
- **Time**: The sequencing variable

A **sampling plan** extends this by explicitly defining expected factor levels, time points (T), and observations per cell (N). This enables:
1. Detection of SDS 5 and 6 (which require expected vs observed comparison)
2. Rich design reports showing what's missing
3. Documentation of experimental intent

## Overview

| SDS | Name | Structure | Recommended Chart | Plan Required? |
|-----|------|-----------|-------------------|----------------|
| 1 | Full Replication | All cells n >= 2 | Xbar-S | No (but valuable) |
| 2 | No Replication | All cells n = 1 | Xbar (MR-based) | No (but valuable) |
| 3 | Partial Replication | Mixed n=1 and n>=2 | Xbar-S (hybrid) | No (but valuable) |
| 4 | Single Stream | One condition over time | IMR | No |
| 5 | Nested Design | Hierarchical structure | IMR | **Yes** |
| 6 | Incomplete Grid | Irregular collection | IMR | **Yes** |

In [ ]:
from processbehavior import ProcessBehavior
from processbehavior.datasets.synthetic import make_sds

---
## SDS 1: Full Replication

Every (factor × time) cell has 2+ observations. This is the ideal structure for VAS.

**Capabilities:**
- Exact within-cell variance estimation
- All VAS residuals (R2-R5)
- Full interaction analysis
- Xbar-S charts with exact limits

In [ ]:
# Generate SDS 1 data
df1 = make_sds(1, K1=3, K2=2, T=8, n_min=2, n_max=5, seed=42)
print(f"Shape: {df1.shape}")
df1.head()

In [ ]:
# Define sampling plan - documents experimental design
plan1 = {
    'factors': {
        'factor 1': ['F1_1', 'F1_2', 'F1_3'],
        'factor 2': ['F2_1', 'F2_2']
    },
    'T': 8,
    'N': 5  # Maximum expected per cell
}

pb1 = ProcessBehavior(df1)
study1 = pb1.formulate(response='y', time='time', plan=plan1)

print(f"SDS: {study1.sds} - {study1.sds_name}")
print(f"Valid charts: {study1.valid_charts}")
print(f"Recommended: {study1.recommended_chart}")
print(f"Residual charts: {study1.residual_charts}")

In [ ]:
# Design report shows plan vs observed
design1 = study1.design()
print(design1)

In [ ]:
# Execute and visualize
result1 = study1.execute()
result1.plot().show()

---
## SDS 2: No Replication

Every (factor × time) cell has exactly 1 observation. Common in automated measurement systems.

**Capabilities:**
- Variance estimated via 2-point moving average
- Xbar analysis with MR-based limits
- Main effects analysis available

In [ ]:
# Generate SDS 2 data
df2 = make_sds(2, K1=3, K2=2, T=10, seed=42)
print(f"Shape: {df2.shape}")

In [ ]:
plan2 = {
    'factors': {
        'factor 1': ['F1_1', 'F1_2', 'F1_3'],
        'factor 2': ['F2_1', 'F2_2']
    },
    'T': 10,
    'N': 1  # Unreplicated by design
}

pb2 = ProcessBehavior(df2)
study2 = pb2.formulate(response='y', time='time', plan=plan2)

print(f"SDS: {study2.sds} - {study2.sds_name}")
print(f"Valid charts: {study2.valid_charts}")

In [ ]:
result2 = study2.execute()
result2.plot().show()

---
## SDS 3: Partial Replication

Mix of n=1 and n>=2 cells. Most common in real-world data collection.

**Capabilities:**
- Hybrid variance estimation (exact for n>1, zero for n=1)
- Xbar-S analysis with hybrid limits

In [ ]:
# Generate SDS 3 data: 50% replicated
df3 = make_sds(3, K1=3, K2=2, T=8, p_replicated=0.5, n_when_replicated=3, seed=42)
print(f"Shape: {df3.shape}")

# Check cell sizes
cell_counts = df3.groupby(['factor 1', 'factor 2', 'time']).size()
print(f"Cell sizes: {cell_counts.value_counts().to_dict()}")

In [ ]:
plan3 = {
    'factors': {
        'factor 1': ['F1_1', 'F1_2', 'F1_3'],
        'factor 2': ['F2_1', 'F2_2']
    },
    'T': 8,
    'N': 3  # Expected when replicated
}

pb3 = ProcessBehavior(df3)
study3 = pb3.formulate(response='y', time='time', plan=plan3)

print(f"SDS: {study3.sds} - {study3.sds_name}")
print(f"Valid charts: {study3.valid_charts}")

In [ ]:
result3 = study3.execute()
result3.plot().show()

---
## SDS 4: Single Stream Over Time

One condition measured repeatedly over time. Classic time series monitoring.

**Capabilities:**
- Perfect for time series monitoring
- All 8 WECO rules applicable
- No factor comparisons (only one level)

In [ ]:
# Generate SDS 4 data: single stream
df4 = make_sds(4, T=40, seed=42)
print(f"Shape: {df4.shape}")
df4.head()

In [ ]:
# No plan needed for single stream (no factors)
pb4 = ProcessBehavior(df4)
study4 = pb4.formulate(response='y', time='time')

print(f"SDS: {study4.sds} - {study4.sds_name}")
print(f"Valid charts: {study4.valid_charts}")
print(f"Recommended: {study4.recommended_chart}")

In [ ]:
result4 = study4.execute()
result4.plot().show()

---
## SDS 5: Nested Design

Hierarchical factor structure where factor 2 levels are nested within factor 1.

**Example:** Filling heads nested within production lines.

**Why a plan is required:** ProcessBehavior needs to know the expected nesting structure to detect that Line1_Head1 belongs to Line1, not Line2.

**Capabilities:**
- Stratified IMR analysis
- Per-stratum control limits

In [ ]:
# Generate SDS 5 data: 2 lines, 3 heads per line
df5 = make_sds(5, L=2, H_per_L=3, T=8, p_active=0.8, seed=42)
print(f"Shape: {df5.shape}")
print(f"Unique factor 2 values: {df5['factor 2'].unique()}")

In [ ]:
# Plan with nested structure - REQUIRED for SDS 5 detection
# Notice: factor 2 levels indicate which factor 1 they belong to
plan5 = {
    'factors': {
        'factor 1': ['Line1', 'Line2'],
        'factor 2': ['Line1_Head1', 'Line1_Head2', 'Line1_Head3',
                     'Line2_Head1', 'Line2_Head2', 'Line2_Head3']
    },
    'T': 8
}

pb5 = ProcessBehavior(df5)
study5 = pb5.formulate(response='y', time='time', plan=plan5)

print(f"SDS: {study5.sds} - {study5.sds_name}")
print(f"Valid charts: {study5.valid_charts}")

In [ ]:
# Design report shows nesting structure
design5 = study5.design()
print(design5)

In [ ]:
result5 = study5.execute()
print(f"Is stratified: {result5.is_stratified}")
print(f"Strata: {result5.strata}")
result5.plot().show()

---
## SDS 6: Incomplete Grid

Irregular or sporadic data collection with less than 75% grid coverage.

**Why a plan is required:** ProcessBehavior needs to know the *expected* grid size to calculate coverage. Without knowing what was expected, it can't determine what's missing.

**Key insight:** The plan defines what *should* exist. The data shows what *does* exist. SDS 6 is detected when observed/expected < 75%.

**Capabilities:**
- Stratified IMR analysis
- Per-stratum control limits
- Results should be interpreted with caution

In [ ]:
# Generate SDS 6 data: incomplete sampling
df6 = make_sds(6, seed=42)
print(f"Shape: {df6.shape}")
print(f"Time range: {df6['time'].min()} - {df6['time'].max()}")
print(f"Unique times observed: {df6['time'].nunique()}")

In [ ]:
# Plan specifies EXPECTED structure - REQUIRED for SDS 6 detection
# This tells ProcessBehavior: "We expected 6 machines × 80 time points"
plan6 = {
    'factors': {
        'factor 1': ['Machine1', 'Machine2', 'Machine3'],
        'factor 2': ['F2_1', 'F2_2']
    },
    'T': 80  # Expected time points
}

pb6 = ProcessBehavior(df6)
study6 = pb6.formulate(response='y', time='time', plan=plan6)

print(f"SDS: {study6.sds} - {study6.sds_name}")
print(f"Valid charts: {study6.valid_charts}")

In [ ]:
# Design report reveals the incomplete coverage
design6 = study6.design()
print(design6)

# Calculate coverage percentage
coverage = design6.R_observed / design6.R * 100
print(f"\nGrid coverage: {coverage:.1f}% (< 75% → SDS 6)")

In [ ]:
result6 = study6.execute()
print(f"Is stratified: {result6.is_stratified}")
print(f"Strata: {result6.strata}")
result6.plot().show()

---
## Working with Stratified Results

For stratified analyses (SDS 5, 6, or any IMR with multiple factor levels), you can drill down to specific subgroups using the `focus()` method:

In [ ]:
# Focus on a single stratum
if result6.is_stratified:
    stratum = result6.strata[0]
    focused = result6.focus(stratum)
    print(f"Focused on: {stratum}")
    focused.plot().show()

---
## Exporting to Excel

All results can be exported to Excel with embedded charts, summary statistics, and full datasets:

In [ ]:
# Export any result to Excel
result1.to_excel(
    'sds1_analysis.xlsx',
    include_summary=True,
    include_charts=True,
    include_chart_images=True,
    include_residuals=True,
    include_full_dataset=True
)
print("Exported to sds1_analysis.xlsx")

---
## Summary

### SDS Detection Summary

| SDS | Detection Basis | Plan Role |
|-----|-----------------|----------|
| 1 | All cells n ≥ 2 | Documents design, enables rich reports |
| 2 | All cells n = 1 | Documents design, enables rich reports |
| 3 | Mixed cell sizes | Documents design, shows sparse cells |
| 4 | Single factor level | Minimal (no factors to document) |
| 5 | Nested hierarchy | **Required** - defines nesting structure |
| 6 | <75% grid coverage | **Required** - defines expected grid |

### Key Takeaways

1. **VAS formulation** always includes: unit of analysis, response, factors, time
2. **Sampling plans** extend formulation with expected factor levels, T, and N
3. **Plans unlock SDS 5/6** by enabling expected vs observed comparison
4. **Best practice**: Always use plans for rigorous analysis and reproducibility
5. **Stratified results** support `focus()` for drilling into specific subgroups